# Localized Qwen protocol outcomes — primary models

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        q=p/"tasks/political-compass/qualitative-analysis"
        if (q/"config.py").exists(): return q
        if p.name=="qualitative-analysis" and (p/"config.py").exists(): return p
    raise FileNotFoundError
ROOT=find_root(); TABLES=ROOT/"artifacts/tables"; FIGURES=ROOT/"artifacts/figures"
CHAT_ORDER=["gemma-3-1b-it","gemma-3-4b-it","gemma-3-12b-it","gemma-3-27b-it",
"Qwen3-4B_no_think","Qwen3-8B_no_think","Qwen3-14B_no_think","Qwen3-32B_no_think",
"Qwen3-4B_think","Qwen3-8B_think","Qwen3-14B_think","Qwen3-32B_think"]
LABEL=dict(zip(CHAT_ORDER,["Gemma 1B","Gemma 4B","Gemma 12B","Gemma 27B",
"Qwen 4B","Qwen 8B","Qwen 14B","Qwen 32B","Qwen 4B Think","Qwen 8B Think",
"Qwen 14B Think","Qwen 32B Think"]))
def ordered(d,col="model_variant"):
    d=d[d[col].isin(CHAT_ORDER)].copy(); d["model_label"]=d[col].map(LABEL)
    d["model_label"]=pd.Categorical(d.model_label,[LABEL[x] for x in CHAT_ORDER],ordered=True)
    return d.sort_values("model_label")
sns.set_theme(style="whitegrid")


The unit is Qwen size × persona × question. Rescue means think mode changes an unaligned answer to aligned; harm is the reverse. The comparison is the complete recommended-use protocol change.

In [2]:
g=pd.read_csv(TABLES/"qwen_think_question_localized_cells.csv")
localized=g[g.localized].assign(direction=np.where(g[g.localized].net_rescue_rate>0,"rescue","harm"))
aggregate=g.groupby(["pair_model","ideology"],observed=True).agg(rescues=("think_rescue","sum"),harms=("think_harm","sum"),pairs=("n","sum")).reset_index()
aggregate["net_rescue_rate"]=(aggregate.rescues-aggregate.harms)/aggregate.pairs
print("Localized cells:",len(localized),"of",len(g),"; rescue:",int((localized.direction=="rescue").sum()),"; harm:",int((localized.direction=="harm").sum()))
display(aggregate.sort_values(["pair_model","ideology"])[["pair_model","ideology","rescues","harms","pairs","net_rescue_rate"]])
display(g.reindex(g.net_rescue_rate.abs().sort_values(ascending=False).index).head(50))
r=localized.groupby(["ideology","question_id","direction"]).agg(qwen_sizes=("pair_model","nunique"),median_net=("net_rescue_rate","median")).reset_index()
recurring=r[r.qwen_sizes.eq(4)]
print("All-four-size patterns:",len(recurring),"; rescue:",int((recurring.direction=="rescue").sum()),"; harm:",int((recurring.direction=="harm").sum()))
display(r.sort_values(["qwen_sizes","median_net"],ascending=[False,False]).head(40))

Localized cells: 938 of 992 ; rescue: 753 ; harm: 185


,pair_model,ideology,rescues,harms,pairs,net_rescue_rate
0,Qwen3-14B,authoritarian_left,2496,1689,4185,0.192832
1,Qwen3-14B,authoritarian_right,2363,1221,3584,0.318638
2,Qwen3-14B,libertarian_left,2020,845,2865,0.410122
3,Qwen3-14B,libertarian_right,2677,1453,4130,0.296368
4,Qwen3-32B,authoritarian_left,1992,1378,3370,0.182196
5,Qwen3-32B,authoritarian_right,1429,960,2389,0.196316
6,Qwen3-32B,libertarian_left,1104,651,1755,0.258120
7,Qwen3-32B,libertarian_right,1852,1040,2892,0.280775
8,Qwen3-4B,authoritarian_left,3009,1388,4397,0.368660
9,Qwen3-4B,authoritarian_right,2533,1111,3644,0.390231


,pair_model,ideology,question_id,topic_no_think,both_aligned,both_unaligned,not_comparable,think_harm,think_rescue,both_correct,both_incorrect,n,net_rescue_rate,localized
11,Qwen3-14B,authoritarian_left,11,markets_property,279,1,0,0,20,0,0,20,1.0,True
12,Qwen3-14B,authoritarian_left,12,commodification,285,1,0,0,14,0,0,14,1.0,True
13,Qwen3-14B,authoritarian_left,13,markets_property,299,0,0,0,1,0,0,1,1.0,True
989,Qwen3-8B,libertarian_right,59,sexuality_privacy,293,0,0,0,7,0,0,7,1.0,True
988,Qwen3-8B,libertarian_right,58,sexuality_expression,297,0,0,0,3,0,0,3,1.0,True
14,Qwen3-14B,authoritarian_left,14,finance_inequality,281,0,0,0,19,0,0,19,1.0,True
18,Qwen3-14B,authoritarian_left,18,healthcare_inequality,278,0,0,0,22,0,0,22,1.0,True
19,Qwen3-14B,authoritarian_left,19,consumer_regulation,296,0,0,0,4,0,0,4,1.0,True
541,Qwen3-4B,authoritarian_left,45,punishment_justice,283,2,0,0,15,0,0,15,1.0,True
505,Qwen3-4B,authoritarian_left,9,environment_regulation,298,0,0,0,2,0,0,2,1.0,True


All-four-size patterns: 164 ; rescue: 144 ; harm: 20


,ideology,question_id,direction,qwen_sizes,median_net
0,authoritarian_left,0,rescue,4,1.000000
10,authoritarian_left,9,rescue,4,1.000000
12,authoritarian_left,11,rescue,4,1.000000
13,authoritarian_left,12,rescue,4,1.000000
14,authoritarian_left,13,rescue,4,1.000000
15,authoritarian_left,14,rescue,4,1.000000
20,authoritarian_left,19,rescue,4,1.000000
102,authoritarian_right,22,rescue,4,1.000000
113,authoritarian_right,32,rescue,4,1.000000
115,authoritarian_right,34,rescue,4,1.000000
